# Подготовка датасета для претрейна

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("salex-slices.csv", header=None, delimiter=";", names=["id", "slices", "natoms", "e_per_atom"])
df

,id,slices,natoms,e_per_atom
0,1306120,La La La La In In Sn Sn Sn Sn Rh Rh Rh Rh Rh R...,20.0,-6.272686
1,2611940,Ga Ga Ga Ga Ni Ni Ni Ni Pt Pt Pt Pt Pt Pt Pt P...,20.0,-5.614831
2,2611947,Cd Cd Cd Cd Cd Cd Cd Cd Cd Cd Cd Cd Ag Ag Au A...,20.0,-1.878980
3,1306121,La La La La In In Sn Sn Sn Sn Rh Rh Rh Rh Rh R...,20.0,-6.264138
4,122,Ag Ag Ag Ag Ag Ag Ag Ag Ag Ag Ag Ag Te Te Te T...,20.0,-2.999581
...,...,...,...,...
451699,6471440,Pb Hf Hf Se 0 1 o o o 0 1 + o o 0 1 o + o 0 2 ...,4.0,-7.105467
451700,6261193,F F F F Rb Rb V 0 6 o o + 0 4 o o o 0 5 o o o ...,7.0,-5.105946
451701,1323248,Sr Sr Sr Zn Zn In Au Au Au Au Au Au 0 11 - o o...,12.0,-2.936802
451702,6685301,Na Na Br Ag 0 3 - - o 0 3 - o o 0 3 o - o 0 3 ...,4.0,-2.429143


In [3]:
df_filtered = df[(df.natoms >= 2) & (df.natoms <= 20) & (df.e_per_atom < -2)]
df_filtered = df_filtered.drop_duplicates(subset=['id'], keep='first')
df_filtered

,id,slices,natoms,e_per_atom
0,1306120,La La La La In In Sn Sn Sn Sn Rh Rh Rh Rh Rh R...,20.0,-6.272686
1,2611940,Ga Ga Ga Ga Ni Ni Ni Ni Pt Pt Pt Pt Pt Pt Pt P...,20.0,-5.614831
3,1306121,La La La La In In Sn Sn Sn Sn Rh Rh Rh Rh Rh R...,20.0,-6.264138
4,122,Ag Ag Ag Ag Ag Ag Ag Ag Ag Ag Ag Ag Te Te Te T...,20.0,-2.999581
9,123,Ag Ag Ag Ag Ag Ag Ag Ag Ag Ag Ag Ag Te Te Te T...,20.0,-2.997118
...,...,...,...,...
451698,7528197,Ni Ca Ca Cu 0 2 o o o 0 2 o + o 0 2 + o o 0 2 ...,4.0,-3.497263
451699,6471440,Pb Hf Hf Se 0 1 o o o 0 1 + o o 0 1 o + o 0 2 ...,4.0,-7.105467
451700,6261193,F F F F Rb Rb V 0 6 o o + 0 4 o o o 0 5 o o o ...,7.0,-5.105946
451701,1323248,Sr Sr Sr Zn Zn In Au Au Au Au Au Au 0 11 - o o...,12.0,-2.936802


In [4]:
df_filtered["size_bin"] = pd.cut(df_filtered["natoms"], bins=[2,4,8,12,16,20], right=False)

sampled = []
target_total = 150_000
counts = df_filtered["size_bin"].value_counts()

for bin_interval, cnt in counts.items():
    frac = cnt / counts.sum()
    n_take = int(np.round(frac * target_total))
    pool = df_filtered[df_filtered["size_bin"] == bin_interval]
    sampled.append(pool.sample(n=min(n_take, len(pool)), random_state=42))
df_sampled = pd.concat(sampled).reset_index(drop=True)
print("Итоговый объём:", len(df_sampled))

Итоговый объём: 149999


In [ ]:
df_sampled

,id,slices,natoms,e_per_atom,size_bin
0,4534414,Pr Pr Co Os Os Os 0 4 o o - 0 4 + + o 0 4 + o ...,6.0,-8.444615,"[4, 8)"
1,6867237,Eu Si Pm Pm 0 2 o o o 0 2 + o o 0 2 o + o 0 3 ...,4.0,-6.489429,"[4, 8)"
2,9087901,Ag Pt Pt Rh 0 2 o o o 0 2 o + o 0 2 + o o 0 2 ...,4.0,-5.499292,"[4, 8)"
3,9090546,Mn Y Y Zn 0 2 o o o 0 2 o + o 0 2 + o o 0 2 + ...,4.0,-5.921079,"[4, 8)"
4,8373927,Au Au Cd Cu 0 2 - o o 0 2 o o + 0 2 o o o 0 1 ...,4.0,-2.930013,"[4, 8)"
...,...,...,...,...,...
149994,3048026,Ce Ce Ho Ho Ho Ho Ho Ho Ho Ho Ho Ho Er Er Er E...,16.0,-4.699753,"[16, 20)"
149995,2401479,Na Na Na Na Na Na Cd Cd Cd Cd Sn Sn Sn Sn Sn S...,16.0,-2.354378,"[16, 20)"
149996,9797506,Ho Ho Fe H H H H H H H H H H H H Rh Rh Rh 0 5 ...,18.0,-4.625807,"[16, 20)"
149997,2825243,K K K K K K Bi Bi Bi Bi Bi Bi Bi Bi Pb Pb Pb P...,18.0,-3.100039,"[16, 20)"


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

In [7]:
vectorizer = TfidfVectorizer(token_pattern=r"(?u)\S+")
tfidf_matrix = vectorizer.fit_transform(df_filtered["slices"])

In [ ]:
K = 50
if tfidf_matrix.shape[0] < K:
    K = tfidf_matrix.shape[0]

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(tfidf_matrix)

df_filtered["cluster"] = cluster_labels
target_size = 150_000
rows_per_cluster = int(np.floor(target_size / K))

sampled_dfs = []
for cluster_id in range(K):
    cluster_subset = df_filtered[df_filtered["cluster"] == cluster_id]
    n_available = len(cluster_subset)
    n_take = min(n_available, rows_per_cluster)
    if n_take > 0:
        sampled = cluster_subset.sample(n=n_take, random_state=42)
        sampled_dfs.append(sampled)

df_clustered = pd.concat(sampled_dfs).reset_index(drop=True)

current_size = len(df_clustered)
if current_size < target_size:
    deficit = target_size - current_size

    cluster_sizes = df_filtered["cluster"].value_counts().sort_values(ascending=False)
    for cluster_id, size in cluster_sizes.items():
        if deficit <= 0:
            break

        already_taken = len(sampled_dfs[cluster_id]) if cluster_id < len(sampled_dfs) else 0
        available = size - already_taken
        if available <= 0:
            continue
        take_now = min(available, deficit)
        extra_samples = df_filtered[
            (df_filtered["cluster"] == cluster_id) & 
            (~df_filtered.index.isin(sampled_dfs[cluster_id].index))
        ].sample(n=take_now, random_state=42)
        sampled_dfs[cluster_id] = pd.concat([sampled_dfs[cluster_id], extra_samples])
        deficit -= take_now
    df_sampled = pd.concat(sampled_dfs).reset_index(drop=True)

print("Итого строк:", len(df_clustered))

display(df_clustered)


In [ ]:
from pathlib import Path

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
slices_txt_path = output_dir / "omat_slices.txt"

df_clustered["slices"].to_csv(slices_txt_path, index=False, header=False)

# Pre-train: синтаксис и семантика SLICES

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import (
    GPT2TokenizerFast,
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
)
import pandas as pd

slices_txt   = "data/processed/omat.txt"

dataset = load_dataset("text", data_files=str(slices_txt))["train"]

split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_slices(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        # padding="max_length",
    )

train_tok = train_ds.map(tokenize_slices, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tokenize_slices, batched=True, remove_columns=["text"])

# train_tok = train_tok.rename_column("input_ids", "labels")
# val_tok   = val_tok.rename_column("input_ids", "labels")
# def add_labels(batch):
#     batch["labels"] = batch["input_ids"].copy()
#     return batch

# train_tok = train_tok.map(add_labels, batched=True)
# val_tok   = val_tok.map(add_labels, batched=True)

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))

# def data_collator(batch):
#     input_ids = torch.stack([torch.tensor(ex["labels"]) for ex in batch])
#     attention_mask = torch.stack([torch.tensor(ex["attention_mask"]) for ex in batch])
#     return {
#         "input_ids":      input_ids,
#         "attention_mask": attention_mask,
#         "labels":         input_ids.clone(),
#     }
# data_collator = DataCollatorWithPadding(tokenizer, padding="longest", return_tensors="pt")
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False, return_tensors="pt"
)

training_args = TrainingArguments(
    output_dir="checkpoints/pretrain_gpt2_slices",
    overwrite_output_dir=True,

    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    warmup_steps=500,
    weight_decay=0.01,

    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=500,
    save_steps=500,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    fp16=True,
    logging_dir="logs/pretrain",

    optim="adamw_torch_fused",
    dataloader_pin_memory=True,
    dataloader_num_workers=8,
    # gradient_checkpointing=False,

)

from transformers import EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    callbacks=[early_stopping],
)

trainer.train(resume_from_checkpoint=True)
# trainer.train()

2025-06-06 21:30:13.679874: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-06 21:30:13.768315: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-06 21:30:13.768374: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-06 21:30:13.771853: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-06 21:30:13.788025: I tensorflow/core/platform/cpu_feature_guar

Generating train split: 0 examples [00:00, ? examples/s]

Train size: 129137
Validation size: 6797


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/129137 [00:00<?, ? examples/s]

Map:   0%|          | 0/6797 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been us

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import (
    GPT2TokenizerFast,
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import pandas as pd

slices_txt   = "data/processed/omat.txt"

dataset = load_dataset("text", data_files=str(slices_txt))["train"]

split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_slices(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )

train_tok = train_ds.map(tokenize_slices, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tokenize_slices, batched=True, remove_columns=["text"])

train_tok = train_tok.rename_column("input_ids", "labels")
val_tok   = val_tok.rename_column("input_ids", "labels")

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))

def data_collator(batch):
    input_ids = torch.stack([torch.tensor(ex["labels"]) for ex in batch])
    attention_mask = torch.stack([torch.tensor(ex["attention_mask"]) for ex in batch])
    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         input_ids.clone(),
    }
# data_collator = DataCollatorWithPadding(tokenizer, padding="longest", return_tensors="pt")

training_args = TrainingArguments(
    output_dir="checkpoints/pretrain_gpt2_slices_2",
    num_train_epochs=4,
    per_device_train_batch_size=6,
    per_device_eval_batch_size=6,
    gradient_accumulation_steps=4,

    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps= 2000,

    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=500,
    save_total_limit=10,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    prediction_loss_only=False,

    fp16=True,
    # gradient_checkpointing=True,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    logging_dir="logs/pretrain_2",

)

from transformers import EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    callbacks=[early_stopping],
)


trainer.train()

print(trainer.state.global_step)

2025-06-07 07:28:58.030161: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-07 07:28:58.098691: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-07 07:28:58.098728: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-07 07:28:58.100298: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-07 07:28:58.110791: I tensorflow/core/platform/cpu_feature_guar

Train size: 129137
Validation size: 6797


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warni

Step,Training Loss,Validation Loss


### Эксперименты

In [ ]:
# ЭТОТ ЭТОТ ЭТОТ ЭТОТ ЭТОТ ЭТОТ ЭТОТ ЭТОТ ЭТОТ ЭТОТ ЭТОТ ЭТОТ ЭТОТ #
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import (
    GPT2TokenizerFast,
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import pandas as pd

slices_txt   = "data/processed/omat.txt"

dataset = load_dataset("text", data_files=str(slices_txt))["train"]

split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_slices(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )

train_tok = train_ds.map(tokenize_slices, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tokenize_slices, batched=True, remove_columns=["text"])

train_tok = train_tok.rename_column("input_ids", "labels")
val_tok   = val_tok.rename_column("input_ids", "labels")

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))

def data_collator(batch):
    input_ids = torch.stack([torch.tensor(ex["labels"]) for ex in batch])
    attention_mask = torch.stack([torch.tensor(ex["attention_mask"]) for ex in batch])
    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         input_ids.clone(),
    }
# data_collator = DataCollatorWithPadding(tokenizer, padding="longest", return_tensors="pt")

training_args = TrainingArguments(
    output_dir="checkpoints/pretrain_gpt2_slices_2",
    num_train_epochs=4,
    per_device_train_batch_size=6,
    per_device_eval_batch_size=6,
    gradient_accumulation_steps=4,

    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps= 2000,

    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=500,
    save_total_limit=10,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    prediction_loss_only=False,

    fp16=True,
    # gradient_checkpointing=True,

    dataloader_num_workers=16,
    dataloader_pin_memory=True,
    dataloader_persistent_workers=True,
    # prefetch_factor=4,

    optim="adamw_torch_fused",

    logging_dir="logs/pretrain_2",

)

from transformers import EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    callbacks=[early_stopping],
)
trainer.train()

print(trainer.state.global_step)

2025-06-07 07:50:48.311577: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-07 07:50:48.379056: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-07 07:50:48.379110: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-07 07:50:48.380661: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-07 07:50:48.391253: I tensorflow/core/platform/cpu_feature_guar

Train size: 129137
Validation size: 6797


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warni

Step,Training Loss,Validation Loss


In [ ]:
import transformers, accelerate
print("transformers version:", transformers.__version__)
print("accelerate   version:", accelerate.__version__)

transformers version: 4.38.0
accelerate   version: 1.7.0


# Finetune

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    GPT2TokenizerFast,
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

PROJECT_ROOT = Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data" / "processed"

TRAIN_CSV = DATA_DIR / "MP20_train_with_slices.csv"
VAL_CSV   = DATA_DIR / "MP20_val_with_slices.csv"

assert TRAIN_CSV.exists(), f"Файл {TRAIN_CSV} не найден"
assert VAL_CSV.exists(),   f"Файл {VAL_CSV} не найден"

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

for col in ["band_gap", "formation_energy", "slices_string"]:
    assert col in train_df.columns, f"В train_df отсутствует колонка {col}"
    assert col in val_df.columns,   f"В val_df отсутствует колонка {col}"

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

bg_tokens = [f"<BG_{i/10:.1f}>" for i in range(0, 101)]
fe_tokens = [f"<FE_{(i-50)/10:+.1f}>" for i in range(0, 101)]

basic_specials = ["<BOS>", "<EOS>", "|"]

all_specials = basic_specials + bg_tokens + fe_tokens

tokenizer.add_special_tokens({"additional_special_tokens": all_specials})

tokenizer.pad_token = tokenizer.eos_token

class SlicesConditionalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer: GPT2TokenizerFast, max_length: int = 512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        bg_val = float(row["band_gap"])
        fe_val = float(row["formation_energy"])

        i_bg = round(bg_val / 0.1)
        i_fe = round((fe_val + 5.0) / 0.1)

        bg_tok = f"<BG_{i_bg/10:.1f}>"
        fe_tok = f"<FE_{(i_fe-50)/10:+.1f}>"

        slices_txt = row["slices_string"].strip()
        text = f"<BOS> {bg_tok} {fe_tok} | {slices_txt} <EOS>"

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        input_ids      = enc["input_ids"].squeeze(0)       
        attention_mask = enc["attention_mask"].squeeze(0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": input_ids.clone()
        }

train_dataset = SlicesConditionalDataset(train_df, tokenizer, max_length=512)
val_dataset   = SlicesConditionalDataset(val_df,   tokenizer, max_length=512)

PRETRAINED_CKPT = "checkpoints/pretrain_gpt2_slices/checkpoint-12000"
model = GPT2LMHeadModel.from_pretrained(PRETRAINED_CKPT)

model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

training_args = TrainingArguments(
    output_dir="checkpoints/finetune_gpt2_slices",
    overwrite_output_dir=True,

    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,

    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,

    evaluation_strategy="steps",
    eval_steps=1000,
    logging_steps=500,

    save_steps=1000,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,
    logging_dir="logs/finetune",
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=3)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=None,
    tokenizer=tokenizer,
    callbacks=[early_stopping],
)
trainer.train()

best_ckpt = trainer.state.best_model_checkpoint
if best_ckpt is None:
    best_ckpt = training_args.output_dir

tokenizer.save_pretrained(best_ckpt)
model.save_pretrained(best_ckpt)
print("Fine-tuning is done. Best checkpoint:", best_ckpt)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


<BOS> <BG_0.7> <FE_-3.0> | Ba Eu Gd Sb O O O O O O 0 8 o o o 0 8 o o + 0 6 o o o 0 6 o + o 0 5 o o o 0 5 + o o 0 7 o o + 0 7 + o o 0 4 o o + 0 4 o + o 0 9 o + o 0 9 + o o 1 7 o - o 1 7 o o o 1 5 o - o 1 5 o o - 1 4 - o o 1 4 o o o 1 8 - o o 1 8 o - o 1 6 - o o 1 6 o o - 1 9 o o - 1 9 o o o 2 8 - - o 2 6 - o - 2 4 - o o 2 5 o - - 2 7 o - o 2 9 o o - 3 9 o o o 3 7 o o o 3 5 o o o 3 4 o o o 3 6 o o o 3 8 o o o <EOS>
<BOS> <BG_0.0> <FE_-0.5> | Ag Ag Au Au O O O O 0 4 o - - 0 5 o - o 0 7 o o - 0 6 o o o 1 6 - o o 1 7 - o o 1 5 o - o 1 4 o - o 2 5 o o o 2 4 o o o 2 6 o o o 2 7 o o o 3 7 - o - 3 6 - o o 3 4 o o - 3 5 o o o <EOS>
<BOS> <BG_5.0> <FE_-3.5> | Rb Rb Hf Hf Cd Cd F F F F F F F F F F F F F F 0 16 - - - 0 7 - - o 0 18 o o - 0 12 - o - 0 9 o o o 0 8 - - - 0 13 o - o 0 19 - - o 0 6 o o - 0 17 o o o 1 7 - - o 1 16 - - o 1 9 o o o 1 18 o o o 1 15 - o o 1 14 o - o 1 19 - - o 1 8 - - o 1 17 o o o 1 6 o o o 2 6 o o o 2 10 o o o 2 15 o o o 2 12 o o o 2 16 o - o 2 18 + o o 2 8 o o o 3 9 o o o 

/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


<BOS> <BG_1.8> <FE_-2.4> | Cs Cs Co Co W W O O O O O O F F F F F F 0 13 o o o 0 8 o o - 0 8 o o o 0 14 o - o 0 14 o o o 0 10 - o o 0 10 o o o 0 11 o - o 0 11 o o - 0 17 - o o 0 17 o o - 0 12 o o o 0 7 o o o 0 16 - o o 0 16 o - o 0 6 o o o 0 15 o o o 0 9 o o o 1 15 o o o 1 15 o + o 1 8 o o o 1 6 o o + 1 6 o + o 1 14 o o o 1 7 o o o 1 7 + o o 1 13 o + o 1 13 + o o 1 12 o o + 1 12 + o o 1 11 o o o 1 10 o o o 1 17 o o o 1 9 o o o 1 9 o o + 1 16 o o o 2 12 + - o 2 16 o - o 2 6 o o o 2 11 + - - 2 13 + o - 2 17 o o - 3 16 - o o 3 7 o o o 3 15 - + o 3 14 o o o 3 10 - + o 3 13 o + o 4 17 - o o 4 10 - o + 4 9 - o + 4 8 o o o 4 7 o o o 4 12 o o + 5 11 o o - 5 14 o o o 5 8 o + - 5 9 o o o 5 15 o + - 5 6 o + o <EOS>
<BOS> <BG_0.0> <FE_-0.4> | Ho Tm In In 0 2 o o o 0 2 o o + 0 2 + o o 0 2 o + o 0 1 o o + 0 1 o + o 0 1 o + + 0 1 + o o 0 1 + o + 0 1 + + o 0 3 - o o 0 3 o - o 0 3 o o - 0 3 o o o 1 3 - - - 1 3 - - o 1 3 o - - 1 3 - o - 1 2 - o o 1 2 o - o 1 2 o o - 1 2 o o o 2 3 - - o 2 3 - o - 2 3 - o 

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    GPT2TokenizerFast,
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

PROJECT_ROOT = Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data" / "processed"

TRAIN_CSV = DATA_DIR / "MP20_train_with_slices.csv"
VAL_CSV   = DATA_DIR / "MP20_val_with_slices.csv"

assert TRAIN_CSV.exists(), f"Файл {TRAIN_CSV} не найден"
assert VAL_CSV.exists(),   f"Файл {VAL_CSV} не найден"

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

for col in ["band_gap", "formation_energy", "slices_string"]:
    assert col in train_df.columns, f"В train_df отсутствует колонка {col}"
    assert col in val_df.columns,   f"В val_df отсутствует колонка {col}"

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

bg_tokens = [f"<BG_{i/10:.1f}>" for i in range(0, 101)]
fe_tokens = [f"<FE_{(i-50)/10:+.1f}>" for i in range(0, 101)]

basic_specials = ["<BOS>", "<EOS>", "<PAD>", "|"]

all_specials = basic_specials + bg_tokens + fe_tokens

special_tokens_dict = {
    "bos_token": "<BOS>",
    "eos_token": "<EOS>",
    "pad_token": "<PAD>",
    "additional_special_tokens": all_specials
}
tokenizer.add_special_tokens(special_tokens_dict)

print("→ BOS  token:", tokenizer.bos_token, tokenizer.bos_token_id)
print("→ EOS  token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("→ PAD  token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("→ '|'  token id:", tokenizer.convert_tokens_to_ids("|"))
print("→ Словарь после расширения:", len(tokenizer))

class SlicesConditionalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer: GPT2TokenizerFast, max_length: int = 512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        bg_tok = f"<BG_{float(row['band_gap']):.1f}>"
        fe_tok = f"<FE_{float(row['formation_energy']):+.1f}>"

        slices_txt = row["slices_string"].strip()
        text = f"{tokenizer.bos_token} {bg_tok} {fe_tok} | {slices_txt} {tokenizer.eos_token}"

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        input_ids      = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         input_ids.clone()
        }

train_dataset = SlicesConditionalDataset(train_df, tokenizer, max_length=512)
val_dataset   = SlicesConditionalDataset(val_df,   tokenizer, max_length=512)

print("Train size:", len(train_dataset))
print("Val   size:", len(val_dataset))

PRETRAINED_CKPT = str(PROJECT_ROOT / "notebooks" / "checkpoints" / "pretrain_gpt2_slices" / "checkpoint-12000")
model = GPT2LMHeadModel.from_pretrained(PRETRAINED_CKPT)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

emb_size = model.get_input_embeddings().weight.shape[0]
print("После resize — embedding size:", emb_size, " / токенов:", len(tokenizer))

training_args = TrainingArguments(
    output_dir=str(PROJECT_ROOT / "checkpoints" / "finetune_gpt2_slices"),
    overwrite_output_dir=True,

    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    warmup_steps=200,
    weight_decay=0.01,

    evaluation_strategy="steps",
    eval_steps=1200,
    logging_steps=100,

    save_steps=1200,
    save_total_limit=4,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,
    logging_dir=str(PROJECT_ROOT / "logs" / "finetune"),
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    callbacks=[early_stopping],
)
trainer.train()

best_ckpt = trainer.state.best_model_checkpoint
if best_ckpt is None:
    best_ckpt = training_args.output_dir

print("Fine-tuning is done. Best checkpoint:", best_ckpt)
tokenizer.save_pretrained(best_ckpt)
model.save_pretrained(best_ckpt)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


→ BOS  token: <BOS> 50257
→ EOS  token: <EOS> 50258
→ PAD  token: <PAD> 50259
→ '|'  token id: 91
→ Словарь после расширения: 50462
Train size: 30716
Val   size: 7680
После resize — embedding size: 50462  / токенов: 50462


/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss
1200,0.249800,0.222682
2400,0.239000,0.215189
3600,0.235800,0.212030
4800,0.230300,0.210826


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Fine-tuning завершён. Лучший чекпоинт: /root/Crystal-generator-using-SLICES/checkpoints/finetune_gpt2_slices/checkpoint-4800
